# Evidence Retrieval Walkthrough

Short interactive path over the `evidence_retrieval` package: build (or load) a
**tiny** local index, run one query, print ranked hits.

For metrics, ablations, and the full case study, see the root **README** and
`python scripts/run_retrieval_eval.py`.

**Not a fact-checker.** ISOT labels are source buckets; nearest neighbors ≠ verification.

## Build or load a tiny index

Uses a small article sample and `data/retrieval_index/tiny`. Needs ISOT CSVs in
`data/` (`python scripts/download_data.py`) and a one-time MiniLM download on first build.

In [ ]:
from pathlib import Path
from evidence_retrieval.index import IndexConfig, PassageIndex

INDEX_DIR = Path("data/retrieval_index/tiny")

if (INDEX_DIR / "chunks.parquet").exists():
    index = PassageIndex.load(INDEX_DIR)
    print(f"Loaded tiny index: {len(index.chunks):,} passages")
else:
    config = IndexConfig(n_articles=200, chunk_words=120, fields=("body",))
    index = PassageIndex.build(data_dir="data", config=config, show_progress=True)
    index.save(INDEX_DIR)
    print(f"Built + saved tiny index → {INDEX_DIR} ({len(index.chunks):,} passages)")

## One query → ranked hits

In [ ]:
query = "Federal Reserve interest rate decision and markets"
hits = index.query_df(query, top_k=5, method="hybrid")
cols = ["rank", "score", "label_name", "title", "passage"]
print(f"query: {query!r} | method: hybrid | hits: {len(hits)}")
print(hits[cols].to_string(index=False, max_colwidth=72))

## CLI equivalent

```bash
python -m evidence_retrieval build --n-articles 200 --out data/retrieval_index/tiny
python -m evidence_retrieval query "Federal Reserve interest rate decision and markets" \
  --index data/retrieval_index/tiny --top-k 5
```